In [32]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

Chroma Vs Pinecorn
- Chroma : 로컬메모리 DB, 인메모리 DB
- Pinecone : 클라우드 vector DB
    (pinecone console에 api key 생성 -> .env 추가) : PINECONE_API_KEY

# 0. 패키지 설치

In [ ]:
# %pip install pinecone-client langchain-pinecone

# 1. Knowledg Base 구성을 위한 데이터 생성

In [33]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# loader = Docx2txtLoader("./tax_docs/소득세법(법률)(제20615호)(20250701).docx")
loader = Docx2txtLoader("./tax_docs/with_table.docx")
splitter = RecursiveCharacterTextSplitter(chunk_size = 1500, chunk_overlap = 200)
document_list = loader.load_and_split(text_splitter=splitter)

In [34]:
len(document_list)

183

In [35]:
# embedding : Upstae API text-embedding-3-large
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(model = "embedding-query")

In [36]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
# 데이터를 처음 저장(업로드)할 때
index_name = "upstage-tax-index"
database = PineconeVectorStore.from_documents(
    documents = document_list,
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 17.5 s
Wall time: 1min 14s


# 1-1. PineconeDB에서 Load

In [6]:
# embedding : OpenAI API text-embedding-3-large
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(model = "embedding-query")

In [7]:
index_name = "upstage-tax-index"

In [8]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
database = PineconeVectorStore(
    embedding=embedding, # query 임베딩, 유사도 검색
    index_name=index_name
)

# 2. 답변 생성을 위한 Retrieval

In [9]:
query = "연봉 5000만원인 거주자의 소득세는 얼마인가요?"
# retrieved_docs = database.similarity_search_with_score (query, k=10,) # 튜플형태로 유사도 수치를 같이 반환
# print(retrieved_docs[0][0].page_content)
# print(retrieved_docs[0][1])
retrieved_docs = database.similarity_search(query, k=10,)

In [10]:
print(retrieved_docs[0].page_content)

③ 거주자의 부양가족 중 거주자(그 배우자를 포함한다)의 직계존속이 주거 형편에 따라 별거하고 있는 경우에는 제1항에도 불구하고 제50조에서 규정하는 생계를 같이 하는 사람으로 본다.

④ 제50조, 제51조 및 제59조의2에 따른 공제대상 배우자, 공제대상 부양가족, 공제대상 장애인 또는 공제대상 경로우대자에 해당하는지 여부의 판정은 해당 과세기간의 과세기간 종료일 현재의 상황에 따른다. 다만, 과세기간 종료일 전에 사망한 사람 또는 장애가 치유된 사람에 대해서는 사망일 전날 또는 치유일 전날의 상황에 따른다.<개정 2014. 1. 1.>

⑤ 제50조제1항제3호 및 제59조의2에 따라 적용대상 나이가 정해진 경우에는 제4항 본문에도 불구하고 해당 과세기간의 과세기간 중에 해당 나이에 해당되는 날이 있는 경우에 공제대상자로 본다.<개정 2014. 1. 1.>

[전문개정 2009. 12. 31.]



제54조(종합소득공제 등의 배제) ① 분리과세이자소득, 분리과세배당소득, 분리과세연금소득과 분리과세기타소득만이 있는 자에 대해서는 종합소득공제를 적용하지 아니한다. <개정 2013. 1. 1.>

② 제70조제1항, 제70조의2제2항 또는 제74조에 따라 과세표준확정신고를 하여야 할 자가 제70조제4항제1호에 따른 서류를 제출하지 아니한 경우에는 기본공제 중 거주자 본인에 대한 분(分)과 제59조의4제9항에 따른 표준세액공제만을 공제한다. 다만, 과세표준확정신고 여부와 관계없이 그 서류를 나중에 제출한 경우에는 그러하지 아니하다.<개정 2013. 1. 1., 2014. 1. 1.>

③ 제82조에 따른 수시부과 결정의 경우에는 기본공제 중 거주자 본인에 대한 분(分)만을 공제한다.

[전문개정 2009. 12. 31.]

[제목개정 2014. 1. 1.]



제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업

In [11]:
retriever = database.as_retriever(
#     search_kwargs = {"k":4}
)
retriever.invoke(query)

[Document(id='9604a190-4b23-4aad-a54f-92f002d21b9c', metadata={'source': './tax_docs/소득세법(법률)(제20615호)(20250701).docx'}, page_content='③ 거주자의 부양가족 중 거주자(그 배우자를 포함한다)의 직계존속이 주거 형편에 따라 별거하고 있는 경우에는 제1항에도 불구하고 제50조에서 규정하는 생계를 같이 하는 사람으로 본다.\n\n④ 제50조, 제51조 및 제59조의2에 따른 공제대상 배우자, 공제대상 부양가족, 공제대상 장애인 또는 공제대상 경로우대자에 해당하는지 여부의 판정은 해당 과세기간의 과세기간 종료일 현재의 상황에 따른다. 다만, 과세기간 종료일 전에 사망한 사람 또는 장애가 치유된 사람에 대해서는 사망일 전날 또는 치유일 전날의 상황에 따른다.<개정 2014. 1. 1.>\n\n⑤ 제50조제1항제3호 및 제59조의2에 따라 적용대상 나이가 정해진 경우에는 제4항 본문에도 불구하고 해당 과세기간의 과세기간 중에 해당 나이에 해당되는 날이 있는 경우에 공제대상자로 본다.<개정 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n\n\n제54조(종합소득공제 등의 배제) ① 분리과세이자소득, 분리과세배당소득, 분리과세연금소득과 분리과세기타소득만이 있는 자에 대해서는 종합소득공제를 적용하지 아니한다. <개정 2013. 1. 1.>\n\n② 제70조제1항, 제70조의2제2항 또는 제74조에 따라 과세표준확정신고를 하여야 할 자가 제70조제4항제1호에 따른 서류를 제출하지 아니한 경우에는 기본공제 중 거주자 본인에 대한 분(分)과 제59조의4제9항에 따른 표준세액공제만을 공제한다. 다만, 과세표준확정신고 여부와 관계없이 그 서류를 나중에 제출한 경우에는 그러하지 아니하다.<개정 2013. 1. 1., 2014. 1. 1.>\n\n③ 제82조에 따른 수시부과 결정의 경우에는 기본공제 중 거주자 본인에 대한 분(分)만을 공제한다.\n\n[전문개정 

# 3. 제공되는 prompt를 활용하여 답변 생성

In [12]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

In [13]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-4.1-nano")

In [14]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm = llm,
    retriever = retriever,
    chain_type_kwargs={"prompt":prompt},
)

In [15]:
ai_message = qa_chain.invoke({"query":query})

print(ai_message)

{'query': '연봉 5000만원인 거주자의 소득세는 얼마인가요?', 'result': '연봉 5000만원인 거주자의 기본 공제는 약 150만원이며, 소득세율은 점진적이어서 대략 6-15% 범위입니다. 공제 후 과세표준에 따라 세율이 적용되고, 이를 기반으로 세액이 산출됩니다. 정확한 세금액은 상세 계산이 필요하지만, 대략 300만 원 내외일 것으로 예상됩니다.'}
